# Iteration 02 - Encode Paper Parameters

## Aim

Encode the base-scenario paper parameters explicitly in Python, covering arrival rates, length-of-stay summaries, routing matrices, and base bed capacities.


## Prompt Used

Let's move on to iteration 02: encode the paper parameters explicitly, starting with arrival rates, LOS tables, and routing matrices.

Additional source provided during the iteration:

- Figure 2 image containing all mean inter-arrival times for acute admission and transfer from elsewhere to rehab.


## Sources Used

- `docs/source_text/Monks_et_al.md`
- `docs/source_text/Monks_et_al_appendix.md`
- Figure 2 image supplied in the chat during Iteration 02


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
src = root / 'src'
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from stroke_sim.parameters import (
    ACUTE_ARRIVAL_MEAN_DAYS,
    BASE_CAPACITY_BEDS,
    LENGTH_OF_STAY,
    REHAB_EXTERNAL_ARRIVAL_MEAN_DAYS,
    ROUTING_FROM_ACUTE,
    ROUTING_FROM_REHAB,
)
from stroke_sim.patients import PatientGroup
from stroke_sim.config import Ward

print('Loaded parameter registry successfully')


## Acute Arrival Means

These values come directly from Figure 2.


In [ ]:
for group, param in ACUTE_ARRIVAL_MEAN_DAYS.items():
    print(group, param.mean_interarrival_days, param.source_note)


## Rehab External Arrival Means

These values come directly from Figure 2. TIA is listed as N/A and is therefore omitted from the explicit stream registry.


In [ ]:
for group, param in REHAB_EXTERNAL_ARRIVAL_MEAN_DAYS.items():
    print(group, param.mean_interarrival_days, param.source_note)


## Length Of Stay Tables

These values come from the appendix LOS summary tables and are stored as reported summaries rather than already-converted lognormal parameters.


In [ ]:
for ward, table in LENGTH_OF_STAY.items():
    print('\n', ward)
    for name, summary in table.items():
        print(name, 'mean=', summary.mean, 'sd=', summary.sd, 'median=', summary.median)


## Routing Matrices

These values come from the appendix transfer matrices.


In [ ]:
print('From acute')
for group, row in ROUTING_FROM_ACUTE.items():
    print(group, row, 'sum=', sum(row.values()))

print('\nFrom rehab')
for group, row in ROUTING_FROM_REHAB.items():
    print(group, row, 'sum=', sum(row.values()))


## Base Capacities


In [ ]:
print(BASE_CAPACITY_BEDS)
assert BASE_CAPACITY_BEDS[Ward.ACUTE] == 10
assert BASE_CAPACITY_BEDS[Ward.REHAB] == 12


## Tests And Checks

Key checks for this iteration:

- acute arrival means match Figure 2
- rehab external arrivals match Figure 2
- LOS tables match the appendix values used in code
- routing rows are internally checked
- the published rehab routing row for `other` is preserved as printed even though it sums to 101%


In [ ]:
assert ACUTE_ARRIVAL_MEAN_DAYS[PatientGroup.STROKE].mean_interarrival_days == 1.2
assert ACUTE_ARRIVAL_MEAN_DAYS[PatientGroup.TIA].mean_interarrival_days == 9.3
assert REHAB_EXTERNAL_ARRIVAL_MEAN_DAYS[PatientGroup.STROKE].mean_interarrival_days == 21.8
assert LENGTH_OF_STAY[Ward.ACUTE]['stroke_esd'].mean == 4.6
assert abs(sum(ROUTING_FROM_REHAB[PatientGroup.OTHER].values()) - 1.01) < 1e-9
print('Iteration 02 checks passed')


## Tester Notes

- Figure 2 was required to recover the full arrival-rate table cleanly.
- The appendix rehab routing row for `other` appears rounded or inconsistent because 13% + 88% = 101%.
- This iteration preserves published values and records the discrepancy rather than silently correcting it.
